In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("data/clean/ae_clean.csv", parse_dates=["month"])
ae = df[["Org Code", "Org name", "Parent Org", "month",
         "total_attendances", "over_4hrs", "waits_12hrs", "has_major_ae"]]
ae = ae.rename(columns={"Org Code": "org_code", "Org name": "org_name", "Parent Org": "region"})

con = sqlite3.connect(":memory:")
ae.to_sql("ae", con, index=False)

def sql(query):
    return pd.read_sql(query, con)

In [2]:
sql("""
SELECT month,
       SUM(total_attendances) AS attendances,
       ROUND(100.0 * (SUM(total_attendances) - SUM(over_4hrs)) / SUM(total_attendances), 1) AS pct_within_4hrs
FROM ae
GROUP BY month
ORDER BY month
""")

,month,attendances,pct_within_4hrs
0,2025-04-01 00:00:00,2297421,74.8
1,2025-05-01 00:00:00,2397529,75.4
2,2025-06-01 00:00:00,2351581,75.6
3,2025-07-01 00:00:00,2408866,76.4
4,2025-08-01 00:00:00,2266937,75.9
5,2025-09-01 00:00:00,2310222,75.1
6,2025-10-01 00:00:00,2401266,74.2
7,2025-11-01 00:00:00,2347046,74.2
8,2025-12-01 00:00:00,2326139,73.8
9,2026-01-01 00:00:00,2319296,72.5


In [3]:
sql("""
SELECT org_code,
       MAX(org_name) AS trust,
       ROUND(100.0 * (SUM(total_attendances) - SUM(over_4hrs)) / SUM(total_attendances), 1) AS pct_within_4hrs
FROM ae
WHERE has_major_ae = 1
GROUP BY org_code
HAVING pct_within_4hrs < 60
ORDER BY pct_within_4hrs
""")

,org_code,trust,pct_within_4hrs
0,RJN,EAST CHESHIRE NHS TRUST,48.4
1,RXW,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,52.0
2,RD1,ROYAL UNITED HOSPITALS BATH NHS FOUNDATION TRUST,58.1
3,RWA,HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST,58.8
4,RTP,SURREY AND SUSSEX HEALTHCARE NHS TRUST,59.4
5,RX1,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,59.4


In [4]:
sql("""
WITH trust_year AS (
    SELECT region,
           org_code,
           MAX(org_name) AS trust,
           ROUND(100.0 * (SUM(total_attendances) - SUM(over_4hrs)) / SUM(total_attendances), 1) AS pct_within_4hrs
    FROM ae
    WHERE has_major_ae = 1
    GROUP BY region, org_code
)
SELECT *
FROM (
    SELECT region, trust, pct_within_4hrs,
           RANK() OVER (PARTITION BY region ORDER BY pct_within_4hrs DESC) AS rank_in_region
    FROM trust_year
)
WHERE rank_in_region = 1
ORDER BY pct_within_4hrs DESC
""")

,region,trust,pct_within_4hrs,rank_in_region
0,NHS ENGLAND NORTH EAST AND YORKSHIRE,SHEFFIELD CHILDREN'S NHS FOUNDATION TRUST,92.9,1
1,NHS ENGLAND NORTH WEST,ALDER HEY CHILDREN'S NHS FOUNDATION TRUST,85.2,1
2,NHS ENGLAND EAST OF ENGLAND,WEST HERTFORDSHIRE TEACHING HOSPITALS NHS TRUST,84.6,1
3,NHS ENGLAND LONDON,HOMERTON HEALTHCARE NHS FOUNDATION TRUST,81.7,1
4,NHS ENGLAND SOUTH WEST,DORSET COUNTY HOSPITAL NHS FOUNDATION TRUST,81.2,1
5,NHS ENGLAND MIDLANDS,BIRMINGHAM WOMEN'S AND CHILDREN'S NHS FOUNDATI...,81.0,1
6,NHS ENGLAND SOUTH EAST,MAIDSTONE AND TUNBRIDGE WELLS NHS TRUST,80.1,1
